# LLM Diagnostic Bias — Interactive Exploration

This notebook provides interactive exploration of experiment results.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Add project root to path
import sys
sys.path.insert(0, '..')

from src.parse_responses import parse_all_results
from src.analyze import run_full_analysis, print_summary
from src.visualize import generate_all_figures

sns.set_theme(style='whitegrid')
%matplotlib inline

## Load Results

After running the experiment, load and parse the results:

In [ ]:
# Parse raw results into structured DataFrame
# Update the path to your specific experiment file
results_path = '../data/results/experiment_latest.json'

df = parse_all_results(results_path)
df.head()

## Overview Statistics

In [ ]:
print(f'Total parsed responses: {len(df)}')
print(f'Conditions: {df["condition"].nunique()}')
print(f'Models: {df["model"].nunique()}')
print(f'Sex groups: {df["patient_sex"].unique()}')
print()

# Correct diagnosis rate by sex
print('Correct diagnosis rate by sex:')
print(df.groupby('patient_sex')['correct_diagnosis_found'].mean())
print()

# Mean urgency by sex
print('Mean urgency by sex:')
print(df.groupby('patient_sex')['urgency'].mean())
print()

# Dismissive pattern rate by sex
print('Dismissive pattern rate by sex:')
print(df.groupby('patient_sex')['dismissive_pattern'].mean())

## Run Full Analysis

In [ ]:
analysis_results = run_full_analysis(df)
print_summary(analysis_results)

## Quick Visualizations

In [ ]:
# Correct diagnosis rate by sex and condition
fig, ax = plt.subplots(figsize=(12, 5))
pivot = df.pivot_table(
    values='correct_diagnosis_found',
    index='condition',
    columns='patient_sex',
    aggfunc='mean'
)
pivot.plot(kind='bar', ax=ax, color=['#E07A5F', '#3D405B', '#81B29A'])
ax.set_title('Correct Diagnosis Rate by Sex and Condition', fontweight='bold')
ax.set_ylabel('Rate')
ax.set_ylim(0, 1.1)
ax.legend(title='Patient Sex')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Urgency distribution by sex
fig, ax = plt.subplots(figsize=(10, 5))
for sex, color in [('female', '#E07A5F'), ('male', '#3D405B'), ('unspecified', '#81B29A')]:
    sex_data = df[df['patient_sex'] == sex]['urgency'].dropna()
    ax.hist(sex_data, bins=range(1, 7), alpha=0.5, color=color, label=sex, edgecolor='white')
ax.set_title('Distribution of Urgency Ratings by Sex', fontweight='bold')
ax.set_xlabel('Urgency (1=routine, 5=emergency)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bias indicator count by sex and condition
fig, ax = plt.subplots(figsize=(12, 5))
non_control = df[df['condition'] != 'acute_mi_classic']
pivot = non_control.pivot_table(
    values='bias_indicator_count',
    index='condition',
    columns='patient_sex',
    aggfunc='mean'
)
pivot.plot(kind='bar', ax=ax, color=['#E07A5F', '#3D405B', '#81B29A'])
ax.set_title('Mean Bias Indicator Count by Sex and Condition', fontweight='bold')
ax.set_ylabel('Mean Count')
ax.legend(title='Patient Sex')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Generate Publication Figures

Save all figures to `outputs/figures/`:

In [ ]:
# Save parsed results first
df.to_csv('../data/results/parsed_results.csv', index=False)
analysis_results.to_csv('../outputs/tables/analysis_results.csv', index=False)

# Then generate all publication figures
generate_all_figures(
    analysis_path='../outputs/tables/analysis_results.csv',
    parsed_path='../data/results/parsed_results.csv'
)